# Import Libraries

In [1]:
import os
import csv
import ast
import requests
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import LabelEncoder
pd.set_option('display.max_rows', None)

In [2]:
import pickle

# Get Packet Fields

In [3]:
def get_nested_field_names(field_element):
    nested_field_names = []
    for nested_field in field_element.findall('field'):
        field_name = nested_field.get('name').replace(".", "_")
        if field_name == '':
         #   continue
         field_name = 'nested_empty_field_name'
        nested_field_names.extend([field_name + '_name', field_name + '_showname', field_name + '_size',
                                              field_name + '_pos', field_name + '_show', field_name + '_value'])
        nested_field_names.extend(get_nested_field_names(nested_field))  # Recursively collect nested fields
    return nested_field_names

In [4]:
def get_packet_field_names(proto):
    packet_fields = []

    for field in proto.findall('field'):
        field_name = field.get('name')
        if not field_name:
            field_name = 'empty_field_name'

        field_name = field_name.replace(".", "_")

        packet_fields.extend([
            field_name + '_name',
            field_name + '_showname',
            field_name + '_size',
            field_name + '_pos',
            field_name + '_show',
            field_name + '_value'
        ])

        packet_fields.extend(get_nested_field_names(field))

    return packet_fields

# Align Packet Features

In [5]:
def align_lists(list1, list2, list2_values):
    list2_values_aligned = []
    list2_aligned = []
    for x in list1:
        if x not in list2:
            list2_values_aligned.append('')
        else:
            idx = list2.index(x)
            list2_values_aligned.append(list2_values[idx])
    return list2_values_aligned

# Make Dataframe from Packet Features

In [6]:
def get_nested_fields(field_element):
    nested_fields = []
    for nested_field in field_element.findall('field'):
        #if nested_field.get('name') == '':
        #    continue
        nested_fields.extend([
                nested_field.get('name'),
                nested_field.get('showname'),
                nested_field.get('size'),
                nested_field.get('pos'),
                nested_field.get('show'),
                nested_field.get('value')
            ])
        nested_fields.extend(get_nested_fields(nested_field))  # Recursively collect nested fields
    return nested_fields

In [ ]:
def makeDataframe(xmlfile):
    dataframe_list = []
    tree = ET.parse(xmlfile)
    root = tree.getroot()
    packets = root.findall('packet')

    column_names = []
    values = []
    flag = False

    # -------- First pass: collect column names --------
    for packet in packets:
        for top_proto in packet.findall('proto'):
          if top_proto.get('name') == 'mac-nr':
            for proto in top_proto.iter("proto"):
              for proto in packet.findall('proto'):
                  if proto.get('name') == 'nr-rrc':
                      if proto.get('hide') == 'yes':
                        continue
                      print("TEST")
                      print(proto.get('name'))
                      columns = get_packet_field_names(proto)
                      column_names = sorted(set(column_names).union(set(columns)))
                      for nas_proto in proto.iter("proto"):
                          if nas_proto.get("name") == "nas-5gs":
                            print(nas_proto.get('name'))
                            columns = get_packet_field_names(nas_proto)
                            column_names = sorted(set(column_names).union(set(columns)))
    column_names = sorted(column_names)

    # -------- Second pass: extract values --------
    for packet in packets:
        for top_proto in packet.findall('proto'):
          if top_proto.get('name') == 'mac-nr':
            for proto in top_proto.iter("proto"):
              for proto in packet.findall('proto'):
                  if proto.get('name') == 'nr-rrc':
                      if proto.get('hide') == 'yes':
                        continue
                      #flag = True
                      packet_fields = []
                      columns = []
                      print("RRC")
                      # extract all fields inside NGAP
                      for field in proto.findall('field'):
                        #if not field.get('name'):
                        #    continue
                        packet_fields.extend([
                            field.get('name'),
                            field.get('showname'),
                            field.get('size'),
                            field.get('pos'),
                            field.get('show'),
                            field.get('value')
                        ])
                        packet_fields.extend(get_nested_fields(field))
                      columns += get_packet_field_names(proto)

                      for nas_proto in proto.iter("proto"):
                        print("NAS")
                        if nas_proto.get("name") == "nas-5gs":
                          for field in nas_proto.findall("field"):
                            packet_fields.extend([
                                field.get('name'),
                                field.get('showname'),
                                field.get('size'),
                                field.get('pos'),
                                field.get('show'),
                                field.get('value')
                            ])
                            packet_fields.extend(get_nested_fields(field))
                          columns += get_packet_field_names(nas_proto)

                      values.append(align_lists(column_names, columns, packet_fields))

    print("Column names:", column_names)
    print("Number of rows:", len(values))
    return list(column_names), values

# Prepare Dataframe from PCAP File

In [8]:
import subprocess

def run_tshark_to_pdml(input_file, xml_output_file):
    # Open the output file in write mode
    with open(xml_output_file, "w") as f:
        # Run tshark - read input_file, output PDML, write stdout to f
        subprocess.run(["tshark", "-r", input_file, "-T", "pdml"], stdout=f, check=True)

In [9]:
def prepare_dataframe_from_pcap_file(xml_output_file):
    #xml_output_file = input_file.replace("pcapng", "xml")
    #run_tshark_to_pdml(input_file, xml_output_file)
    try:
      column_names, values = makeDataframe(xml_output_file)
    except:
      print("error")
    df = pd.DataFrame(values, columns=column_names)
    return df

In [19]:
run_tshark_to_pdml(r"pcap_files/test.pcapng", r"XML_files/test.xml")